# 03 ViT 输入组装：CLS Token 与位置编码

上一课已经实现 Patch Embedding：

```text
B x 3 x 32 x 32
→ B x 64 x 192
```

现在已经有 64 个 patch tokens，但还不能直接照搬标准 ViT 的分类流程。还需要解决两个问题：

1. Encoder 输出 64 个 token，最终应该用哪个 token 完成整张图片分类？
2. Self-Attention 怎样知道某个 patch 来自图片左上角还是右下角？

标准 ViT 分别使用 **CLS token** 和 **位置编码** 解决这两个问题。

## 1. 本课学习目标

完成本课后，需要能够：

1. 解释 CLS token 是什么、为什么加入它。
2. 解释 CLS 在进入 Encoder 前还没有图片信息。
3. 使用 `expand` 为一个 batch 准备 CLS token。
4. 使用 `torch.cat` 把 CLS 拼到 patch tokens 前面。
5. 解释为什么 Self-Attention 需要位置信息。
6. 正确创建 `1 x (N+1) x D` 的可学习位置编码。
7. 区分 CLS 的“拼接”和位置编码的“相加”。
8. 实现完整的 `ViTInputEmbedding` 模块。
9. 验证 CLS、位置编码和 Patch Embedding 都能获得梯度。
10. 用 EncoderLayer 观察 CLS 怎样从公共参数变成图片相关表示。

## 2. 当前数据走到哪里了

对于 CIFAR-10 Tiny ViT，当前配置是：

```text
image_size = 32
patch_size = 4
embed_dim = 192
```

Patch Embedding 输出：

$$
X_{patch}:B\times64\times192
$$

其中每张图片有 64 个 patch tokens，每个 token 是 192 维。

这一课要完成：

$$
B\times64\times192
\longrightarrow
B\times65\times192
$$

并让这 65 个 token 同时带有内容信息和位置信息。

## 3. CLS token 是什么

CLS 是 classification 的缩写。CLS token 是模型中的一个可学习向量，它不对应任何真实 patch。

可以先把序列想成：

```text
[CLS] [patch 1] [patch 2] ... [patch 64]
```

CLS 被放在序列第 0 个位置，并和所有 patch tokens 一起经过每一层 Self-Attention。

经过多层 Encoder 后：

```text
patch tokens：保存各个区域经过上下文改写后的信息
CLS token：逐层汇总与整张图片分类有关的信息
```

最终分类头只读取第 0 个位置的 CLS 表示。

### CLS 一开始就是整张图片的摘要吗

不是。进入第一个 EncoderBlock 前，CLS 只是一个与图片无关的可学习参数。

对于同一个 batch 中的不同图片，初始 CLS 内容完全相同：

$$
CLS_1=CLS_2=\cdots=CLS_B
$$

只有经过 Self-Attention 后，CLS 才会读取当前图片的 patch tokens。不同图片的 patches 不同，因此 Encoder 输出的 CLS 也会变得不同。

所以准确说法是：

```text
CLS 不是预先计算好的图片摘要，
而是模型为“学习如何汇总图片”准备的特殊位置。
```

## 4. 准备实验环境和 Patch Embedding

为了让本 Notebook 可以独立运行，保留上一课已经理解过的精简版 `PatchEmbedding`。本课不会再次展开卷积切 patch 的细节。

In [1]:
import torch
from torch import nn

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch：", torch.__version__)
print("计算设备：", device)

PyTorch： 2.10.0
计算设备： cuda


In [2]:
class PatchEmbedding(nn.Module):
    def __init__(self, image_size=32, patch_size=4, in_channels=3, embed_dim=192):
        super().__init__()
        if image_size % patch_size != 0:
            raise ValueError("image_size 必须能被 patch_size 整除")

        self.image_size = image_size
        self.grid_size = image_size // patch_size
        self.num_patches = self.grid_size ** 2
        self.projection = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size, stride=patch_size,
        )

    def forward(self, x):
        _, _, height, width = x.shape
        if height != self.image_size or width != self.image_size:
            raise ValueError(
                f"期望 {self.image_size} x {self.image_size}，"
                f"实际得到 {height} x {width}"
            )
        x = self.projection(x)
        return x.flatten(2).transpose(1, 2)

In [3]:
batch_size = 2
embed_dim = 192
fake_images = torch.randn(batch_size, 3, 32, 32, device=device)
patch_embedding = PatchEmbedding(embed_dim=embed_dim).to(device)
patch_tokens = patch_embedding(fake_images)

print("图片 shape：", tuple(fake_images.shape))
print("patch tokens shape：", tuple(patch_tokens.shape))
print("num_patches：", patch_embedding.num_patches)

图片 shape： (2, 3, 32, 32)
patch tokens shape： (2, 64, 192)
num_patches： 64


## 5. 创建一个可学习 CLS 参数

CLS 参数通常保存为：

$$
1\times1\times D
$$

第一个 1 表示先只保存一份公共参数；第二个 1 表示它只占一个 token 位置；$D$ 必须和 patch token 的特征维度相同。

为什么不是直接保存成 `B x 1 x D`？

因为 batch_size 会变化。模型不能为 batch 中每个位置保存一套独立 CLS 参数。正确做法是保存一份 `1 x 1 x D`，前向传播时临时扩展到当前 batch。

In [4]:
cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim, device=device))
nn.init.trunc_normal_(cls_token, std=0.02)

expanded_cls = cls_token.expand(batch_size, -1, -1)

print("模型中保存的 CLS：", tuple(cls_token.shape))
print("为当前 batch 扩展后：", tuple(expanded_cls.shape))
print("两张图片的初始 CLS 是否相同：", torch.allclose(expanded_cls[0], expanded_cls[1]))

模型中保存的 CLS： (1, 1, 192)
为当前 batch 扩展后： (2, 1, 192)
两张图片的初始 CLS 是否相同： True


### `expand(batch_size, -1, -1)` 怎样理解

当前 CLS 是 `1 x 1 x 192`。

```python
cls_token.expand(batch_size, -1, -1)
```

含义是：

- 第 0 维从 1 扩展到当前 batch_size；
- 两个 `-1` 表示后面的维度保持原大小；
- batch 中所有位置都引用同一组可学习 CLS 参数。

这不是为每张图片创建不同参数。图片之间的差异要等 Self-Attention 读取不同 patch tokens 后才会出现。

## 6. 把 CLS 拼到 patch tokens 前面

当前两个张量是：

$$
CLS:B\times1\times D
$$

$$
PatchTokens:B\times64\times D
$$

需要沿 token 维，也就是 `dim=1` 拼接：

$$
B\times1\times D
+
B\times64\times D
\xrightarrow{cat}
B\times65\times D
$$

batch 维和特征维必须一致，只有 token 数量发生变化。

In [5]:
tokens_with_cls = torch.cat([expanded_cls, patch_tokens], dim=1)

print("patch tokens：", tuple(patch_tokens.shape))
print("CLS token：", tuple(expanded_cls.shape))
print("拼接后：", tuple(tokens_with_cls.shape))
print("第 0 个位置是否就是 CLS：", torch.allclose(tokens_with_cls[:, :1], expanded_cls))
print("后 64 个位置是否仍是 patches：", torch.allclose(tokens_with_cls[:, 1:], patch_tokens))

patch tokens： (2, 64, 192)
CLS token： (2, 1, 192)
拼接后： (2, 65, 192)
第 0 个位置是否就是 CLS： True
后 64 个位置是否仍是 patches： True


## 7. 为什么还要位置编码

Patch Embedding 会按照固定顺序排列 patches，但 Self-Attention 的核心计算只根据 token 内容计算相关性。没有额外位置信息时，它不能可靠地区分：

```text
某个 patch 来自左上角
还是来自右下角
```

图像的空间位置非常重要。例如天空通常位于上方，地面通常位于下方；眼睛、鼻子、嘴巴之间也有空间结构。

因此，需要给序列中的每个位置加入一个位置向量，让 token 同时包含：

```text
patch 内容信息 + patch 位置信息
```

## 8. 可学习位置编码的 shape

加入 CLS 后一共有 $N+1=65$ 个 token，每个 token 是 $D=192$ 维。因此位置编码保存为：

$$
PositionEmbedding:1\times65\times192
$$

65 个位置分别对应：

```text
位置 0：CLS
位置 1：patch 1
位置 2：patch 2
...
位置 64：patch 64
```

最前面的 1 表示所有图片共享同一套位置参数。相加时，PyTorch 会沿 batch 维广播到 `B x 65 x 192`。

In [6]:
sequence_length = patch_embedding.num_patches + 1
position_embedding = nn.Parameter(
    torch.zeros(1, sequence_length, embed_dim, device=device)
)
nn.init.trunc_normal_(position_embedding, std=0.02)

encoder_input = tokens_with_cls + position_embedding

print("tokens_with_cls：", tuple(tokens_with_cls.shape))
print("position_embedding：", tuple(position_embedding.shape))
print("相加后的 Encoder 输入：", tuple(encoder_input.shape))
print("相加前后 shape 是否相同：", tokens_with_cls.shape == encoder_input.shape)

tokens_with_cls： (2, 65, 192)
position_embedding： (1, 65, 192)
相加后的 Encoder 输入： (2, 65, 192)
相加前后 shape 是否相同： True


## 9. 一定要区分：CLS 是拼接，位置编码是相加

这是本课最重要的 shape 区别。

### 加入 CLS

```text
操作：torch.cat(..., dim=1)
效果：增加一个 token 位置
shape：B x 64 x 192 → B x 65 x 192
```

### 加入位置编码

```text
操作：tokens + position_embedding
效果：给已有位置增加位置信息
shape：B x 65 x 192 → B x 65 x 192
```

位置编码不能沿 token 维继续拼接，否则会额外制造 65 个“位置 tokens”，完全改变序列含义。

## 10. 为什么输入组装后常接 Dropout

标准 ViT 通常在 token 内容与位置编码相加后使用 Dropout：

$$
X_0=Dropout([CLS;PatchTokens]+PositionEmbedding)
$$

Dropout 在训练时随机把部分特征置零，起到正则化作用；在 `eval()` 模式下关闭。

Dropout 不改变张量 shape，也没有可学习参数。为了让本课数值实验可重复，后面的模块测试暂时使用 `dropout=0.0`。正式训练时可以再设为 0.1。

## 11. 实现完整的 ViTInputEmbedding

现在把这一课的步骤封装起来：

```text
图片
→ PatchEmbedding
→ 扩展 CLS
→ 沿 dim=1 拼接
→ 加入位置编码
→ Dropout
→ Encoder 输入
```

CLS 和位置编码必须使用 `nn.Parameter`，否则优化器不会把它们当作模型参数更新。

In [7]:
class ViTInputEmbedding(nn.Module):
    def __init__(
        self, image_size=32, patch_size=4,
        in_channels=3, embed_dim=192, dropout=0.1,
    ):
        super().__init__()
        self.patch_embedding = PatchEmbedding(
            image_size=image_size, patch_size=patch_size,
            in_channels=in_channels, embed_dim=embed_dim,
        )
        self.num_patches = self.patch_embedding.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.position_embedding = nn.Parameter(
            torch.zeros(1, self.num_patches + 1, embed_dim)
        )
        self.dropout = nn.Dropout(dropout)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.position_embedding, std=0.02)

    def forward(self, images):
        patch_tokens = self.patch_embedding(images)
        batch_size = patch_tokens.shape[0]
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        tokens = torch.cat([cls_tokens, patch_tokens], dim=1)

        if tokens.shape[1] != self.position_embedding.shape[1]:
            raise ValueError("token 数量与位置编码长度不一致")

        tokens = tokens + self.position_embedding
        return self.dropout(tokens)

In [8]:
vit_input = ViTInputEmbedding(dropout=0.0).to(device)
encoder_input = vit_input(fake_images)

print(vit_input)
print("CLS 参数：", tuple(vit_input.cls_token.shape))
print("位置编码参数：", tuple(vit_input.position_embedding.shape))
print("Encoder 输入：", tuple(encoder_input.shape))
print("num_patches：", vit_input.num_patches)

ViTInputEmbedding(
  (patch_embedding): PatchEmbedding(
    (projection): Conv2d(3, 192, kernel_size=(4, 4), stride=(4, 4))
  )
  (dropout): Dropout(p=0.0, inplace=False)
)
CLS 参数： (1, 1, 192)
位置编码参数： (1, 65, 192)
Encoder 输入： (2, 65, 192)
num_patches： 64


## 12. batch_size 改变时发生什么

模型参数中只保存一份 CLS 和一份位置编码，所以 batch_size 改变时不需要重新创建模型。

应该变化的只有第 0 维 $B$：

```text
2 张图片 → 2 x 65 x 192
5 张图片 → 5 x 65 x 192
```

$N=65$ 和 $D=192$ 都不由 batch_size 决定。

In [9]:
for current_batch_size in [1, 2, 5]:
    current_images = torch.randn(current_batch_size, 3, 32, 32, device=device)
    current_output = vit_input(current_images)
    print(f"batch_size={current_batch_size} → {tuple(current_output.shape)}")

print("模型中 CLS 参数仍是：", tuple(vit_input.cls_token.shape))
print("模型中位置编码仍是：", tuple(vit_input.position_embedding.shape))

batch_size=1 → (1, 65, 192)
batch_size=2 → (2, 65, 192)
batch_size=5 → (5, 65, 192)
模型中 CLS 参数仍是： (1, 1, 192)
模型中位置编码仍是： (1, 65, 192)


## 13. CLS 和位置编码增加多少参数

CLS 参数量：

$$
1\times1\times192=192
$$

位置编码参数量：

$$
1\times65\times192=12480
$$

Patch Embedding 参数量是 9408。因此输入组装模块总参数量：

$$
9408+192+12480=22080
$$

`expand` 和 Dropout 不会增加参数。

In [10]:
print("可学习参数及 shape：")
for name, parameter in vit_input.named_parameters():
    print(f"{name:38s} {tuple(parameter.shape)}")

parameter_count = sum(p.numel() for p in vit_input.parameters())
print("\n总参数量：", parameter_count)

可学习参数及 shape：
cls_token                              (1, 1, 192)
position_embedding                     (1, 65, 192)
patch_embedding.projection.weight      (192, 3, 4, 4)
patch_embedding.projection.bias        (192,)

总参数量： 22080


## 14. 验证三类参数都能获得梯度

输入组装模块中有三类可学习参数：

```text
Patch Embedding 卷积权重
CLS token
位置编码
```

继续使用假损失检查反向传播。只要三类梯度都存在且不是全 0，就说明以后真实分类 loss 可以更新它们。

In [11]:
vit_input.zero_grad()
encoder_input = vit_input(fake_images)
fake_loss = encoder_input.square().mean()
fake_loss.backward()

gradient_records = {
    "Patch Embedding": vit_input.patch_embedding.projection.weight.grad,
    "CLS token": vit_input.cls_token.grad,
    "Position Embedding": vit_input.position_embedding.grad,
}

print("假损失：", fake_loss.item())
for name, gradient in gradient_records.items():
    print(
        f"{name:20s} shape={tuple(gradient.shape)}, "
        f"norm={gradient.norm().item():.6f}, "
        f"all_zero={bool(torch.all(gradient == 0))}"
    )

假损失： 0.33041203022003174
Patch Embedding      shape=(192, 3, 4, 4), norm=0.094016, all_zero=False
CLS token            shape=(1, 1, 192), norm=0.000063, all_zero=False
Position Embedding   shape=(1, 65, 192), norm=0.007358, all_zero=False


## 15. 把组装结果送入一个 EncoderLayer

这一课不展开 EncoderBlock 的内部实现，只验证输入接口是否正确。

我们创建一个符合 Tiny ViT 配置的 EncoderLayer：

```text
d_model = 192
nhead = 3
dim_feedforward = 768
batch_first = True
norm_first = True
```

`batch_first=True` 表示它接收的正是 `B x N x D`。EncoderLayer 不改变序列长度和特征维度，所以输出仍应为 `B x 65 x 192`。

In [12]:
encoder_layer = nn.TransformerEncoderLayer(
    d_model=192,
    nhead=3,
    dim_feedforward=768,
    dropout=0.0,
    batch_first=True,
    norm_first=True,
).to(device)
encoder_layer.eval()

with torch.inference_mode():
    encoder_input = vit_input(fake_images)
    encoder_output = encoder_layer(encoder_input)

print("Encoder 输入：", tuple(encoder_input.shape))
print("Encoder 输出：", tuple(encoder_output.shape))
print("输入输出 shape 是否一致：", encoder_input.shape == encoder_output.shape)

Encoder 输入： (2, 65, 192)
Encoder 输出： (2, 65, 192)
输入输出 shape 是否一致： True


## 16. 观察 CLS 怎样变成图片相关表示

当前 batch 中有两张不同的随机图片。进入 Encoder 前，它们使用相同的 CLS 参数和相同的 CLS 位置编码，所以第 0 个 token 应该完全相同。

进入 Encoder 后，CLS 通过 Self-Attention 读取各自图片的 64 个 patch tokens。两张图片的 patch 内容不同，因此输出 CLS 应该变得不同。

In [13]:
input_cls_equal = torch.allclose(encoder_input[0, 0], encoder_input[1, 0])
output_cls_equal = torch.allclose(encoder_output[0, 0], encoder_output[1, 0])
output_difference = (encoder_output[0, 0] - encoder_output[1, 0]).norm().item()

print("进入 Encoder 前，两张图片的 CLS 是否相同：", input_cls_equal)
print("经过 Encoder 后，两张图片的 CLS 是否相同：", output_cls_equal)
print("两个输出 CLS 的差异范数：", output_difference)

进入 Encoder 前，两张图片的 CLS 是否相同： True
经过 Encoder 后，两张图片的 CLS 是否相同： False
两个输出 CLS 的差异范数： 2.95855450630188


## 17. 小实验：位置编码为什么真的有用

对同一张图片，把 64 个 patch tokens 的顺序随机打乱，但保持 CLS 位于第 0 个位置。

我们比较两种情况：

1. 不加位置编码：Encoder 只看到相同的一组 patch 内容，主要只是排列顺序不同。
2. 加位置编码：打乱 patch 后，相同内容被分配到了不同空间位置。

对于没有位置编码的 Self-Attention，重新排列非 CLS tokens 不应该改变 CLS 对这一组内容的汇总结果；加入位置编码后，位置和内容的对应关系发生变化，CLS 输出会改变。

In [14]:
with torch.inference_mode():
    one_image_patches = vit_input.patch_embedding(fake_images[:1])
    one_cls = vit_input.cls_token.expand(1, -1, -1)
    original_tokens = torch.cat([one_cls, one_image_patches], dim=1)

    permutation = torch.randperm(vit_input.num_patches, device=device)
    shuffled_tokens = torch.cat(
        [one_cls, one_image_patches[:, permutation]], dim=1
    )

    cls_no_pos_original = encoder_layer(original_tokens)[:, 0]
    cls_no_pos_shuffled = encoder_layer(shuffled_tokens)[:, 0]

    cls_with_pos_original = encoder_layer(
        original_tokens + vit_input.position_embedding
    )[:, 0]
    cls_with_pos_shuffled = encoder_layer(
        shuffled_tokens + vit_input.position_embedding
    )[:, 0]

no_position_difference = (cls_no_pos_original - cls_no_pos_shuffled).abs().max().item()
with_position_difference = (cls_with_pos_original - cls_with_pos_shuffled).abs().max().item()

print("不加位置编码时的最大差异：", no_position_difference)
print("加入位置编码后的最大差异：", with_position_difference)
print("不加位置编码时是否近似相同：", torch.allclose(
    cls_no_pos_original, cls_no_pos_shuffled, atol=1e-5
))

不加位置编码时的最大差异： 2.384185791015625e-07
加入位置编码后的最大差异： 0.012014508247375488
不加位置编码时是否近似相同： True


### 怎样理解实验结果

不加位置编码时，两个 CLS 输出只存在浮点计算顺序造成的极小误差，可以认为相同。这说明仅靠 Self-Attention，CLS 知道“有哪些 patch 内容”，却不能可靠知道这些内容原来位于哪里。

加入位置编码后，打乱 patch 会改变“内容 + 位置”的配对，CLS 输出出现明显差异。

这个实验展示的是随机初始化 Encoder 的结构性质，不代表模型已经理解真实空间关系。位置编码提供了学习空间关系所需的条件，真正有用的空间模式仍要通过训练学到。

## 18. patch_size 为什么会影响位置编码长度

对于 32 x 32 图片：

```text
patch_size=4：64 patches + 1 CLS → 位置编码长度 65
patch_size=8：16 patches + 1 CLS → 位置编码长度 17
```

因此，修改 patch_size 不只是修改 Patch Embedding。它还会改变序列长度和位置编码参数 shape。

这也是加载预训练 ViT 并修改输入分辨率时经常需要处理位置编码插值的原因。本阶段先固定图片尺寸和 patch_size，不提前展开插值实现。

## 19. 常见误解

### 误解一：CLS 一加入就包含整张图片信息

不是。初始 CLS 对所有图片相同，经过 Self-Attention 后才吸收当前图片的 patch 信息。

### 误解二：每张图片有一套独立 CLS 参数

不是。模型只学习一份 `1 x 1 x D` 参数，batch 中所有图片共享它。

### 误解三：CLS 和位置编码都使用 cat

不是。CLS 沿 token 维拼接；位置编码与 tokens 逐元素相加。

### 误解四：位置编码只给 patches，不给 CLS

标准 ViT 的可学习位置编码通常包含 CLS 对应的位置 0，因此长度是 $N+1$。

### 误解五：位置编码相加后 token 数量增加

不会。相加前后 shape 保持 `B x (N+1) x D`。

### 误解六：`nn.Parameter` 只是普通 Tensor 的别名

不是。把 Tensor 包装成 `nn.Parameter` 并挂在模块属性上后，它会出现在 `model.parameters()` 和 `state_dict()` 中，优化器才能自动管理它。

## 20. 本课动手任务

### 任务一：独立预测 shape

把 batch_size 改成 8，先写出 patch tokens、扩展 CLS、拼接后序列和位置编码相加后的 shape，再运行核对。

### 任务二：故意沿错误维度 cat

在临时单元尝试 `dim=2`，阅读报错并解释为什么 token 数量维没有增加。完成后删除临时单元。

### 任务三：验证位置编码广播

打印 batch 中两张图片使用的 `position_embedding` shape，解释为什么模型只保存第 0 维为 1 的参数。

### 任务四：修改 patch_size

创建 patch_size=8 的新模块，预测并验证 `num_patches`、CLS shape、位置编码 shape 和输出 shape。

### 任务五：自己复述 CLS 的变化

不用代码，用三句话说明 CLS 在 Encoder 前、经过 Self-Attention 时、进入分类头前分别是什么状态。

## 21. 本节小结

完整输入组装流程是：

```text
图片：B x 3 x 32 x 32
→ Patch Embedding
patch tokens：B x 64 x 192
→ 扩展并拼接 CLS
tokens_with_cls：B x 65 x 192
→ 加入可学习位置编码 1 x 65 x 192
→ Dropout
Encoder 输入：B x 65 x 192
```

CLS 和位置编码的核心分工是：

```text
CLS：为整张图片分类提供一个可学习的信息汇总位置
位置编码：为 CLS 和每个 patch token 提供位置身份
```

下一课将开始实现 Tiny ViT 的 EncoderBlock，重点把 Multi-Head Self-Attention、LayerNorm、残差连接和 FFN 组装成可训练模块。

### 完成本课后的掌握标准

- 能解释初始 CLS 为什么与图片无关；
- 能正确使用 `expand` 和 `cat(dim=1)`；
- 能从 64 个 patches 推出 65 个位置；
- 能写出位置编码的 `1 x 65 x 192` shape；
- 能区分拼接与逐元素相加；
- 能解释位置编码为什么沿 batch 广播；
- 能实现 `ViTInputEmbedding`；
- 能核对参数量和三类参数的梯度；
- 能通过实验解释 CLS 如何变成图片相关表示；
- 能说明 patch_size 与位置编码长度的关系。

## 22. 自测问题

1. CLS token 为什么不对应真实图片 patch？
2. 为什么初始 CLS 参数保存为 `1 x 1 x D`？
3. `expand(B, -1, -1)` 中两个 `-1` 表示什么？
4. 为什么要沿 `dim=1` 拼接 CLS？
5. 64 个 patch tokens 加入 CLS 后为什么是 65 个 tokens？
6. CLS 在什么时候开始包含当前图片的信息？
7. 为什么 Self-Attention 需要额外位置信息？
8. 位置编码为什么是 `1 x 65 x 192`？
9. CLS 和位置编码分别使用 cat 还是加法？
10. 加入位置编码后为什么 shape 不变？
11. batch_size 从 2 变成 5 时，CLS 参数本身的 shape 会变化吗？
12. 为什么 CLS 和位置编码必须使用 `nn.Parameter`？
13. 进入 Encoder 前，不同图片的 CLS 为什么相同？
14. 经过 Encoder 后，不同图片的 CLS 为什么不同？
15. patch_size 从 4 改成 8 后，位置编码长度应该是多少？

### 自测参考答案

1. 它是专门为分类学习的信息汇总位置，不来自任何具体图像区域。
2. 模型只需要保存一份公共 CLS 参数，batch 维在前向传播时动态扩展。
3. 保持 token 数量维和特征维的原大小。
4. 第 1 维是 token 数量维，拼接后要让序列长度增加 1。
5. 64 个 patch 位置前增加了 1 个 CLS 位置。
6. 经过 Self-Attention、读取当前图片的 patch tokens 时。
7. Attention 主要根据内容关系计算，本身不能可靠表示 patch 的二维空间位置。
8. 1 表示一套共享参数，65 表示 CLS 加 64 个 patches，192 是每个位置向量的维度。
9. CLS 使用 `cat(dim=1)`；位置编码使用逐元素加法。
10. 被相加的 token 和位置向量一一对应，三个维度均兼容，不新增位置。
11. 不会，仍然是 `1 x 1 x D`；只有扩展视图变成 `5 x 1 x D`。
12. 这样它们才会注册为模型参数，被优化器和 state_dict 管理。
13. 所有图片共享相同 CLS 参数和 CLS 位置编码，尚未与 patches 交流。
14. 每张图片的 patch 内容不同，Self-Attention 为 CLS 汇总出的信息也不同。
15. 32/8=4，每张图片有 4 x 4=16 个 patches，加 CLS 后长度为 17。